### Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import cdsapi
import zipfile

### Global config

In [2]:
START_DATE = "2005-01-01"
END_DATE   = "2010-12-31"

BENCH_POINTS = pd.DataFrame([
    {"site_id": "FR_PARIS",     "area": "france", "lat": 48.8566, "lon":  2.3522},
    {"site_id": "FR_LYON",      "area": "france", "lat": 45.7640, "lon":  4.8357},
    {"site_id": "FR_TOULOUSE",  "area": "france", "lat": 43.6047, "lon":  1.4442},
    {"site_id": "FR_MARSEILLE", "area": "france", "lat": 43.2965, "lon":  5.3698},

    {"site_id": "BR_BELO_HORIZONTE", "area": "brazil", "lat": -19.9167, "lon": -43.9345},
    {"site_id": "BR_GOV_VALADARES",  "area": "brazil", "lat": -18.8540, "lon": -41.9550},
    {"site_id": "BR_IPATINGA",       "area": "brazil", "lat": -19.4703, "lon": -42.5476},
    {"site_id": "BR_VITORIA",        "area": "brazil", "lat": -20.3155, "lon": -40.3128},
])

CMIP6_MODELS = [
    "CESM2-FV2",
    "MPI-ESM1-2-HR",
    "IPSL-CM6A-LR",
    "GFDL-ESM4",
]

# CMIP6 via CDS
CMIP6_DATASET_ID = "projections-cmip6"
CDS_VAR_MAP = {
    "tas": "near_surface_air_temperature",
    "pr":  "precipitation",
}

# ERA5 timeseries via CDS
ERA5_TS_DATASET_ID = "reanalysis-era5-single-levels-timeseries"
ERA5_VARIABLES = [
    "2m_temperature",
    "total_precipitation",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
]

OUT_ROOT = Path("../../../benchmark_cmip6_vs_era5")
ERA5_DIR = OUT_ROOT / "era5_raw"
CMIP6_DIR = OUT_ROOT / "cmip6_raw"
ERA5_DIR.mkdir(parents=True, exist_ok=True)
CMIP6_DIR.mkdir(parents=True, exist_ok=True)

print("Benchmark period:", START_DATE, "→", END_DATE)
print("Points:", len(BENCH_POINTS), "| FR:", (BENCH_POINTS.area=="france").sum(), "| BR:", (BENCH_POINTS.area=="brazil").sum())
print("CMIP6 models:", CMIP6_MODELS)
print("CMIP6 vars:", list(CDS_VAR_MAP.keys()))

Benchmark period: 2005-01-01 → 2010-12-31
Points: 8 | FR: 4 | BR: 4
CMIP6 models: ['CESM2-FV2', 'MPI-ESM1-2-HR', 'IPSL-CM6A-LR', 'GFDL-ESM4']
CMIP6 vars: ['tas', 'pr']


### Helpers

In [3]:
def extract_first_member(zip_path: Path, extract_dir: Path, suffix: str) -> Path:
    """
    Extract the first file ending with `suffix` from a ZIP into extract_dir.
    """
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        matches = [n for n in z.namelist() if n.lower().endswith(suffix.lower())]
        if not matches:
            raise ValueError(f"No {suffix} inside {zip_path.name}. Contents: {z.namelist()[:30]}")
        name = matches[0]

        z.extract(name, path=extract_dir)
        extracted = extract_dir / name

        out_fp = extract_dir / Path(name).name
        if extracted.exists() and extracted != out_fp:
            out_fp.parent.mkdir(parents=True, exist_ok=True)
            extracted.replace(out_fp)

    return out_fp

def safe_unlink(p: Path):
    try:
        p.unlink()
    except FileNotFoundError:
        return
    except Exception as e:
        print(f"Could not delete {p.name}: {e}")

def years_months_days_from_window(start_date: str, end_date: str):
    start_y = int(start_date[:4])
    end_y   = int(end_date[:4])
    years  = [str(y) for y in range(start_y, end_y + 1)]
    months = [f"{m:02d}" for m in range(1, 13)]
    days   = [f"{d:02d}" for d in range(1, 32)]
    return years, months, days

### ERA5 downloader

In [4]:
def download_era5_timeseries_point(site_id: str, lat: float, lon: float, out_csv: Path) -> Path:
    """
    Download ERA5 hourly timeseries at a single point.
    CDS returns a ZIP: we extract first CSV and rename to out_csv.
    """
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    zip_fp = out_csv.with_suffix(".zip")

    c = cdsapi.Client()
    req = {
        "variable": ERA5_VARIABLES,
        "location": {"latitude": float(lat), "longitude": float(lon)},
        "date": [f"{START_DATE}/{END_DATE}"],
        "data_format": "csv",
    }

    c.retrieve(ERA5_TS_DATASET_ID, req, str(zip_fp.resolve()))

    extracted = extract_first_member(zip_fp, out_csv.parent, ".csv")
    extracted.replace(out_csv)
    safe_unlink(zip_fp)

    return out_csv

era5_csv_paths = {}

for _, r in BENCH_POINTS.iterrows():
    site_id = r["site_id"]
    csv_fp = ERA5_DIR / f"era5_{site_id}_{START_DATE}_{END_DATE}.csv"
    zip_fp = ERA5_DIR / f"era5_{site_id}_{START_DATE}_{END_DATE}.zip"

    if csv_fp.exists():
        era5_csv_paths[site_id] = csv_fp
        print("Exists:", csv_fp.name)
        continue

    if zip_fp.exists():
        extracted = extract_first_member(zip_fp, ERA5_DIR, ".csv")
        extracted.replace(csv_fp)
        safe_unlink(zip_fp)
        era5_csv_paths[site_id] = csv_fp
        print("Extracted existing zip:", zip_fp.name, "->", csv_fp.name)
        continue

    print("Downloading ERA5:", site_id)
    era5_csv_paths[site_id] = download_era5_timeseries_point(site_id, r["lat"], r["lon"], csv_fp)
    print("  saved ->", csv_fp.name)

print("ERA5 CSVs ready:", len(era5_csv_paths))

Exists: era5_FR_PARIS_2005-01-01_2010-12-31.csv
Exists: era5_FR_LYON_2005-01-01_2010-12-31.csv
Exists: era5_FR_TOULOUSE_2005-01-01_2010-12-31.csv
Exists: era5_FR_MARSEILLE_2005-01-01_2010-12-31.csv
Exists: era5_BR_BELO_HORIZONTE_2005-01-01_2010-12-31.csv
Exists: era5_BR_GOV_VALADARES_2005-01-01_2010-12-31.csv
Exists: era5_BR_IPATINGA_2005-01-01_2010-12-31.csv
Exists: era5_BR_VITORIA_2005-01-01_2010-12-31.csv
ERA5 CSVs ready: 8


### Download CMIP6 daily

In [5]:
def to_cds_model_name(model: str) -> str:
    return model.strip().lower().replace("-", "_")

def download_cmip6_daily(model: str, var: str, out_nc: Path) -> Path:
    """
    Download CMIP6 daily historical for one model+var.
    CDS returns ZIP : extract first .nc and rename to out_nc.
    """
    if var not in CDS_VAR_MAP:
        raise ValueError(f"{var=} not found in CDS_VAR_MAP")

    out_nc.parent.mkdir(parents=True, exist_ok=True)
    zip_fp = out_nc.with_suffix(".zip")

    years, months, days = years_months_days_from_window(START_DATE, END_DATE)

    req = {
        "temporal_resolution": "daily",
        "experiment": "historical",
        "variable": CDS_VAR_MAP[var],
        "model": to_cds_model_name(model),
        "year": years,
        "month": months,
        "day": days,
        "format": "zip",
    }

    c = cdsapi.Client()
    c.retrieve(CMIP6_DATASET_ID, req, str(zip_fp.resolve()))

    extracted = extract_first_member(zip_fp, out_nc.parent, ".nc")
    extracted.replace(out_nc)
    safe_unlink(zip_fp)

    return out_nc

cmip6_nc_paths = {}  # (model, var) -> nc_fp

for model in CMIP6_MODELS:
    for var in CDS_VAR_MAP.keys():
        nc_fp = CMIP6_DIR / f"cmip6_{to_cds_model_name(model)}_{var}_{START_DATE}_{END_DATE}.nc"

        if nc_fp.exists():
            cmip6_nc_paths[(model, var)] = nc_fp
            print("Exists:", nc_fp.name)
            continue

        print(f"Downloading CMIP6 | model={model} | var={var} ({CDS_VAR_MAP[var]})")
        try:
            cmip6_nc_paths[(model, var)] = download_cmip6_daily(model, var, nc_fp)
            print("  saved ->", nc_fp.name)
        except Exception as e:
            print(f"Failed model={model} var={var}: {e}")

print("CMIP6 NCs ready:", len(cmip6_nc_paths))

Exists: cmip6_cesm2_fv2_tas_2005-01-01_2010-12-31.nc
Exists: cmip6_cesm2_fv2_pr_2005-01-01_2010-12-31.nc
Exists: cmip6_mpi_esm1_2_hr_tas_2005-01-01_2010-12-31.nc
Exists: cmip6_mpi_esm1_2_hr_pr_2005-01-01_2010-12-31.nc
Exists: cmip6_ipsl_cm6a_lr_tas_2005-01-01_2010-12-31.nc
Exists: cmip6_ipsl_cm6a_lr_pr_2005-01-01_2010-12-31.nc
Exists: cmip6_gfdl_esm4_tas_2005-01-01_2010-12-31.nc
Exists: cmip6_gfdl_esm4_pr_2005-01-01_2010-12-31.nc
CMIP6 NCs ready: 8


### ERA5 hourly CSV to daily (tas_C, pr_mm_day) per site

In [6]:
def read_era5_csv(fp: Path) -> pd.DataFrame:
    """
    ERA5 timeseries CSV from CDS contains a 'time' column and variables.
    parse robustly + normalize column names.
    """
    df = pd.read_csv(fp)

    time_col = None
    for c in df.columns:
        if c.lower() in ("time", "date", "datetime", "valid_time"):
            time_col = c
            break
    if time_col is None:
        raise ValueError(f"No time column found in {fp.name}. Columns: {df.columns.tolist()}")

    df[time_col] = pd.to_datetime(df[time_col], errors="raise")
    df = df.rename(columns={time_col: "time"})

    rename = {
        "2m_temperature": "t2m",
        "total_precipitation": "tp",
        "10m_u_component_of_wind": "u10",
        "10m_v_component_of_wind": "v10",
    }
    for k, v in rename.items():
        if k in df.columns:
            df = df.rename(columns={k: v})

    keep = ["time", "t2m", "tp", "u10", "v10"]
    cols = [c for c in keep if c in df.columns]
    return df[cols].copy()

def era5_hourly_to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert hourly ERA5 to daily:
      tas_C = mean(t2m) in °C
      pr_mm_day = sum(tp) in mm/day  (tp is meters of water)
    """
    out = df.copy()
    out["date"] = out["time"].dt.floor("D")

    # temperature: K to °C, daily mean
    if "t2m" in out.columns:
        out["tas_C"] = out["t2m"] - 273.15

    # precipitation: meters to mm, daily sum
    if "tp" in out.columns:
        out["pr_mm"] = out["tp"] * 1000.0

    agg_dict = {}
    if "tas_C" in out.columns:
        agg_dict["tas_C"] = "mean"
    if "pr_mm" in out.columns:
        agg_dict["pr_mm"] = "sum"

    daily = out.groupby("date", as_index=False).agg(agg_dict)
    daily = daily.rename(columns={"pr_mm": "pr_mm_day"})
    return daily

era5_daily_rows = []
for site_id, csv_fp in era5_csv_paths.items():
    df_h = read_era5_csv(csv_fp)
    df_d = era5_hourly_to_daily(df_h)
    df_d["site_id"] = site_id
    era5_daily_rows.append(df_d)

era5_daily = pd.concat(era5_daily_rows, ignore_index=True)
era5_daily = era5_daily.sort_values(["site_id", "date"]).reset_index(drop=True)

print("ERA5 daily:", era5_daily.shape)
era5_daily.head()


ERA5 daily: (17528, 4)


,date,tas_C,pr_mm_day,site_id
0,2005-01-01,22.427947,19.786358,BR_BELO_HORIZONTE
1,2005-01-02,21.297820,1.261234,BR_BELO_HORIZONTE
2,2005-01-03,21.685935,3.631115,BR_BELO_HORIZONTE
3,2005-01-04,21.446598,7.020473,BR_BELO_HORIZONTE
4,2005-01-05,20.670792,10.861397,BR_BELO_HORIZONTE


### CMIP6 NetCDF to daily at benchmark points (tas_C, pr_mm_day)

In [7]:
def open_cmip6_ds(nc_fp: Path) -> xr.Dataset:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    return xr.open_dataset(nc_fp, decode_times=time_coder)

def cmip6_point_da(ds: xr.Dataset, var: str, lat: float, lon: float) -> xr.DataArray:
    use_360 = float(np.nanmax(ds["lon"].values)) > 180
    lon_sel = (float(lon) % 360) if (use_360 and float(lon) < 0) else float(lon)
    return ds[var].sel(lat=float(lat), lon=lon_sel, method="nearest")

def da_to_daily_series(da: xr.DataArray) -> pd.Series:
    s = da.to_series()
    try:
        s.index = pd.to_datetime(s.index)
    except Exception:
        s.index = xr.CFTimeIndex(s.index).to_datetimeindex(time_unit="ns")
    s.index = pd.to_datetime(s.index).floor("D")
    s = s.groupby(s.index).mean()
    return s

def cmip6_to_common_units(var: str, s: pd.Series) -> pd.Series:
    if var == "tas":
        return s - 273.15
    if var == "pr":
        return s * 86400.0
    return s

cmip6_rows = []
ds_cache = {}  # (model,var) -> xr.Dataset (optional caching)

for (model, var), nc_fp in cmip6_nc_paths.items():
    if (model, var) not in ds_cache:
        ds_cache[(model, var)] = open_cmip6_ds(nc_fp)
    ds = ds_cache[(model, var)]

    if var not in ds.data_vars:
        print(f"Missing {var} in {nc_fp.name} (vars={list(ds.data_vars)[:10]})")
        continue

    out_col = "tas_C" if var == "tas" else ("pr_mm_day" if var == "pr" else var)

    for _, p in BENCH_POINTS.iterrows():
        site_id = p["site_id"]

        da = cmip6_point_da(ds, var, p["lat"], p["lon"])
        s = da_to_daily_series(da)
        s = cmip6_to_common_units(var, s)

        df = s.rename(out_col).reset_index()
        df.columns = ["date", out_col]
        df["site_id"] = site_id
        df["model"] = model
        cmip6_rows.append(df)

cmip6_daily = (
    pd.concat(cmip6_rows, ignore_index=True)
    .groupby(["site_id", "model", "date"], as_index=False)
    .mean(numeric_only=True) 
    .sort_values(["site_id", "model", "date"])
    .reset_index(drop=True)
)

print("CMIP6 daily:", cmip6_daily.shape)
cmip6_daily.head()

C:\Users\nazim\AppData\Local\Temp\ipykernel_2468\1517389902.py:15: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  s.index = xr.CFTimeIndex(s.index).to_datetimeindex(time_unit="ns")
C:\Users\nazim\AppData\Local\Temp\ipykernel_2468\1517389902.py:15: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from the standard calendar.  This may lead to subtle errors in operations that depend on the length of time between dates.
  s.index = xr.CFTimeIndex(s.index).to_datetimeindex(time_unit="ns")
C:\Users\nazim\AppData\Local\Temp\ipykernel_2468\1517389902.py:15: RuntimeWarning: Converting a CFTimeIndex with dates from a non-standard calendar, 'noleap', to a pandas.DatetimeIndex, which uses dates from th

CMIP6 daily: (70096, 5)


,site_id,model,date,tas_C,pr_mm_day
0,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-01,23.771820,2.566234
1,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-02,23.197906,13.947025
2,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-03,22.852051,29.396610
3,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-04,24.103973,5.088216
4,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-05,24.289612,0.783384


### Checks

In [8]:
def qc_table(df: pd.DataFrame, name: str, keys, value_cols):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    print(f"\n=== QC: {name} ===")
    print("shape:", df.shape)
    print("date range:", df["date"].min(), "→", df["date"].max())
    print("duplicate key rows:", int(df.duplicated(list(keys)).sum()))
    print("value cols:", value_cols)
    print("\nmissing rate:")
    print(df[value_cols].isna().mean())

    # rows per group (site,model) or (site)
    group_cols = list(keys[:-1])
    print("\nrows per group describe:")
    print(df.groupby(group_cols).size().describe())

qc_table(cmip6_daily, "CMIP6", keys=("site_id","model","date"), value_cols=["tas_C","pr_mm_day"])
qc_table(era5_daily,  "ERA5",  keys=("site_id","date"),       value_cols=["tas_C","pr_mm_day"])


=== QC: CMIP6 ===
shape: (70096, 5)
date range: 2005-01-01 00:00:00 → 2010-12-31 00:00:00
duplicate key rows: 0
value cols: ['tas_C', 'pr_mm_day']

missing rate:
tas_C        0.0
pr_mm_day    0.0
dtype: float64

rows per group describe:
count      32.000000
mean     2190.500000
std         0.508001
min      2190.000000
25%      2190.000000
50%      2190.500000
75%      2191.000000
max      2191.000000
dtype: float64

=== QC: ERA5 ===
shape: (17528, 4)
date range: 2005-01-01 00:00:00 → 2010-12-31 00:00:00
duplicate key rows: 0
value cols: ['tas_C', 'pr_mm_day']

missing rate:
tas_C        0.0
pr_mm_day    0.0
dtype: float64

rows per group describe:
count       8.0
mean     2191.0
std         0.0
min      2191.0
25%      2191.0
50%      2191.0
75%      2191.0
max      2191.0
dtype: float64


### Align CMIP6 and ERA5

In [9]:
cmip6_daily["date"] = pd.to_datetime(cmip6_daily["date"])
era5_daily["date"]  = pd.to_datetime(era5_daily["date"])

cmp = cmip6_daily.merge(
    era5_daily,
    on=["site_id", "date"],
    how="inner",
    suffixes=("_cmip6", "_era5")
)

print("Aligned comparison table:", cmp.shape)
print("Date range:", cmp["date"].min(), "→", cmp["date"].max())
cmp.head()

Aligned comparison table: (70096, 7)
Date range: 2005-01-01 00:00:00 → 2010-12-31 00:00:00


,site_id,model,date,tas_C_cmip6,pr_mm_day_cmip6,tas_C_era5,pr_mm_day_era5
0,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-01,23.771820,2.566234,22.427947,19.786358
1,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-02,23.197906,13.947025,21.297820,1.261234
2,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-03,22.852051,29.396610,21.685935,3.631115
3,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-04,24.103973,5.088216,21.446598,7.020473
4,BR_BELO_HORIZONTE,CESM2-FV2,2005-01-05,24.289612,0.783384,20.670792,10.861397


### Metrics

In [10]:
def compute_metrics(df, yhat, y, group_cols=("site_id", "model")):
    out = []
    for keys, g in df.groupby(list(group_cols)):
        g = g.dropna(subset=[yhat, y])
        if len(g) == 0:
            continue
        err = g[yhat] - g[y]
        rmse = float(np.sqrt(np.mean(err**2)))
        mae  = float(np.mean(np.abs(err)))
        bias = float(np.mean(err))
        corr = float(np.corrcoef(g[yhat], g[y])[0,1]) if len(g) > 1 else np.nan
        out.append((*keys, len(g), rmse, mae, bias, corr))
        
    cols = list(group_cols) + ["n_days", "rmse", "mae", "bias", "corr"]
    return pd.DataFrame(out, columns=cols)

# Compute metrics
metrics_tas = compute_metrics(cmp, "tas_C_cmip6", "tas_C_era5")
metrics_pr  = compute_metrics(cmp, "pr_mm_day_cmip6", "pr_mm_day_era5")

# Display top results
print("Temperature metrics (tas_C):")
display(metrics_tas.sort_values(["site_id","rmse"]).head(12))

print("\nPrecipitation metrics (pr_mm_day):")
display(metrics_pr.sort_values(["site_id","rmse"]).head(12))

# Ranks
rank_tas = metrics_tas.groupby("model")["rmse"].mean().sort_values()
rank_pr  = metrics_pr.groupby("model")["rmse"].mean().sort_values()

print("\nAvg RMSE (tas_C) by model:")
display(rank_tas)

print("\nAvg RMSE (pr_mm_day) by model:")
display(rank_pr)

Temperature metrics (tas_C):


,site_id,model,n_days,rmse,mae,bias,corr
3,BR_BELO_HORIZONTE,MPI-ESM1-2-HR,2191,2.590963,2.020501,-1.215481,0.620343
2,BR_BELO_HORIZONTE,IPSL-CM6A-LR,2191,2.609547,2.010436,0.233511,0.518898
1,BR_BELO_HORIZONTE,GFDL-ESM4,2190,2.710756,2.108515,-0.152782,0.521187
0,BR_BELO_HORIZONTE,CESM2-FV2,2190,3.254435,2.636167,1.767876,0.666985
4,BR_GOV_VALADARES,CESM2-FV2,2190,3.111024,2.494836,-1.367839,0.636178
6,BR_GOV_VALADARES,IPSL-CM6A-LR,2191,3.786464,3.125239,-2.659105,0.451820
5,BR_GOV_VALADARES,GFDL-ESM4,2190,4.592262,3.986690,-3.688479,0.474184
7,BR_GOV_VALADARES,MPI-ESM1-2-HR,2191,4.717411,4.188278,-4.072332,0.563951
8,BR_IPATINGA,CESM2-FV2,2190,3.086130,2.461065,-1.121647,0.627720
10,BR_IPATINGA,IPSL-CM6A-LR,2191,3.312392,2.683430,-1.881152,0.444159



Precipitation metrics (pr_mm_day):


,site_id,model,n_days,rmse,mae,bias,corr
2,BR_BELO_HORIZONTE,IPSL-CM6A-LR,2191,8.213058,4.468838,0.714302,0.271917
0,BR_BELO_HORIZONTE,CESM2-FV2,2190,8.779298,4.422274,-0.660289,0.136849
3,BR_BELO_HORIZONTE,MPI-ESM1-2-HR,2191,9.616836,4.871848,0.293379,0.132911
1,BR_BELO_HORIZONTE,GFDL-ESM4,2190,11.586941,5.213546,0.317721,0.085081
4,BR_GOV_VALADARES,CESM2-FV2,2190,8.806480,3.807599,-1.469496,0.036093
7,BR_GOV_VALADARES,MPI-ESM1-2-HR,2191,9.497006,4.384173,0.133454,0.130370
6,BR_GOV_VALADARES,IPSL-CM6A-LR,2191,9.671815,4.665613,0.608820,0.130696
5,BR_GOV_VALADARES,GFDL-ESM4,2190,12.047599,5.232896,0.736960,0.036040
8,BR_IPATINGA,CESM2-FV2,2190,8.965121,4.217718,-0.764728,0.072122
11,BR_IPATINGA,MPI-ESM1-2-HR,2191,9.207384,4.392008,0.131270,0.151262



Avg RMSE (tas_C) by model:


model
IPSL-CM6A-LR     3.656905
CESM2-FV2        3.682505
MPI-ESM1-2-HR    4.299910
GFDL-ESM4        4.483218
Name: rmse, dtype: float64


Avg RMSE (pr_mm_day) by model:


model
CESM2-FV2        7.556172
MPI-ESM1-2-HR    8.224193
IPSL-CM6A-LR     8.744130
GFDL-ESM4        9.426518
Name: rmse, dtype: float64

In [11]:
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go


cmp_plot = cmp.copy()
cmp_plot["date"] = pd.to_datetime(cmp_plot["date"])

SITES  = sorted(cmp_plot["site_id"].unique().tolist())
MODELS = sorted(cmp_plot["model"].unique().tolist())

VAR_MAP = {
    "Temperature (°C) — tas_C": ("tas_C_cmip6", "tas_C_era5", "tas_C"),
    "Precip (mm/day) — pr_mm_day": ("pr_mm_day_cmip6", "pr_mm_day_era5", "pr_mm_day"),
}

site_dd  = widgets.Dropdown(options=SITES, value=SITES[0], description="Site:", layout=widgets.Layout(width="320px"))
var_dd   = widgets.Dropdown(options=list(VAR_MAP.keys()), value=list(VAR_MAP.keys())[0], description="Variable:", layout=widgets.Layout(width="360px"))
model_dd = widgets.Dropdown(options=["(all)"] + MODELS, value="(all)", description="Model:", layout=widgets.Layout(width="320px"))

out = widgets.Output()

def make_fig(site_id, var_label, model_choice):
    cmip6_col, era5_col, short = VAR_MAP[var_label]
    d = cmp_plot[cmp_plot["site_id"] == site_id].sort_values("date")

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=d["date"], y=d[era5_col],
        mode="lines", name="ERA5",
        line=dict(width=2)
    ))

    if model_choice == "(all)":
        for m in MODELS:
            dm = d[d["model"] == m]
            fig.add_trace(go.Scatter(
                x=dm["date"], y=dm[cmip6_col],
                mode="lines", name=m,
                opacity=0.75
            ))
        title = f"{site_id} — {short} | ERA5 vs CMIP6 (all models)"
    else:
        dm = d[d["model"] == model_choice]
        fig.add_trace(go.Scatter(
            x=dm["date"], y=dm[cmip6_col],
            mode="lines", name=model_choice,
            line=dict(width=2),
        ))
        title = f"{site_id} — {short} | ERA5 vs {model_choice}"

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title=short,
        hovermode="x unified",
        legend_title="Series",
        height=520,
        margin=dict(l=40, r=20, t=60, b=40),
    )
    return fig

def update_plot(*_):
    with out:
        out.clear_output(wait=True)
        fig = make_fig(site_dd.value, var_dd.value, model_dd.value)
        fig.show()

site_dd.observe(update_plot, names="value")
var_dd.observe(update_plot, names="value")
model_dd.observe(update_plot, names="value")

display(widgets.HBox([site_dd, var_dd, model_dd]))
display(out)

update_plot()

Output()

In [12]:
from tslearn.metrics import soft_dtw

def compute_soft_dtw(df, yhat, y, group_cols=("site_id", "model"), gamma=1.0):
    out = []
    for keys, g in df.groupby(list(group_cols)):
        g = g.dropna(subset=[yhat, y]).sort_values("date")
        if len(g) < 2:
            continue

        ts_pred = g[yhat].to_numpy(dtype=float).reshape(-1, 1)
        ts_true = g[y].to_numpy(dtype=float).reshape(-1, 1)

        dist = float(soft_dtw(ts_pred, ts_true, gamma=gamma))
        out.append((*keys, len(g), dist))

    cols = list(group_cols) + ["n_days", "soft_dtw_dist"]
    return pd.DataFrame(out, columns=cols)

print("Computing Soft DTW for Temperature...")
sdtw_tas = compute_soft_dtw(cmp, "tas_C_cmip6", "tas_C_era5", gamma=1.0)

print("Computing Soft DTW for Precipitation...")
sdtw_pr  = compute_soft_dtw(cmp, "pr_mm_day_cmip6", "pr_mm_day_era5", gamma=1.0)

metrics_tas = metrics_tas.merge(sdtw_tas, on=["site_id", "model", "n_days"], how="left")
metrics_pr  = metrics_pr.merge(sdtw_pr,  on=["site_id", "model", "n_days"], how="left")

display(metrics_tas.sort_values(["site_id", "rmse"]).head(12))
display(metrics_pr.sort_values(["site_id", "rmse"]).head(12))

Computing Soft DTW for Temperature...
Computing Soft DTW for Precipitation...


,site_id,model,n_days,rmse,mae,bias,corr,soft_dtw_dist
3,BR_BELO_HORIZONTE,MPI-ESM1-2-HR,2191,2.590963,2.020501,-1.215481,0.620343,761.294107
2,BR_BELO_HORIZONTE,IPSL-CM6A-LR,2191,2.609547,2.010436,0.233511,0.518898,1569.529089
1,BR_BELO_HORIZONTE,GFDL-ESM4,2190,2.710756,2.108515,-0.152782,0.521187,1866.627343
0,BR_BELO_HORIZONTE,CESM2-FV2,2190,3.254435,2.636167,1.767876,0.666985,3312.520967
4,BR_GOV_VALADARES,CESM2-FV2,2190,3.111024,2.494836,-1.367839,0.636178,1945.391845
6,BR_GOV_VALADARES,IPSL-CM6A-LR,2191,3.786464,3.125239,-2.659105,0.451820,4725.016126
5,BR_GOV_VALADARES,GFDL-ESM4,2190,4.592262,3.986690,-3.688479,0.474184,4875.055690
7,BR_GOV_VALADARES,MPI-ESM1-2-HR,2191,4.717411,4.188278,-4.072332,0.563951,6579.076738
8,BR_IPATINGA,CESM2-FV2,2190,3.086130,2.461065,-1.121647,0.627720,2169.523023
10,BR_IPATINGA,IPSL-CM6A-LR,2191,3.312392,2.683430,-1.881152,0.444159,3417.081201


,site_id,model,n_days,rmse,mae,bias,corr,soft_dtw_dist
2,BR_BELO_HORIZONTE,IPSL-CM6A-LR,2191,8.213058,4.468838,0.714302,0.271917,28590.267722
0,BR_BELO_HORIZONTE,CESM2-FV2,2190,8.779298,4.422274,-0.660289,0.136849,29441.924250
3,BR_BELO_HORIZONTE,MPI-ESM1-2-HR,2191,9.616836,4.871848,0.293379,0.132911,35234.123828
1,BR_BELO_HORIZONTE,GFDL-ESM4,2190,11.586941,5.213546,0.317721,0.085081,78203.140259
4,BR_GOV_VALADARES,CESM2-FV2,2190,8.806480,3.807599,-1.469496,0.036093,48316.624341
7,BR_GOV_VALADARES,MPI-ESM1-2-HR,2191,9.497006,4.384173,0.133454,0.130370,38424.441594
6,BR_GOV_VALADARES,IPSL-CM6A-LR,2191,9.671815,4.665613,0.608820,0.130696,41887.349364
5,BR_GOV_VALADARES,GFDL-ESM4,2190,12.047599,5.232896,0.736960,0.036040,59125.881967
8,BR_IPATINGA,CESM2-FV2,2190,8.965121,4.217718,-0.764728,0.072122,39470.378725
11,BR_IPATINGA,MPI-ESM1-2-HR,2191,9.207384,4.392008,0.131270,0.151262,34975.189762
